# openSEPPO to search NISAR Data and process GCOV Products

Author: Josef Kellndorfer

This example shows the use of openSEPPO tools to search and convert NISAR stacks to COGs, GTiff or simply subset to h5.

For full documentation see https://openseppo.readthedocs.io

In [ ]:
import os
from  openseppo.cli import nisar_search, nisar_gcov_convert

# 1. Data Search 
## Search available data at a point

In [ ]:
LON = -71
LAT = 46
cmd = f'seppo_nisar_search --point {LON} {LAT} --group --https'
print(cmd)

In [ ]:
nisar_search._main(cmd)

## Let's pick track 104 frame 25 and generate a url list output

In [ ]:
track = 104
frame = 25
out = f"{os.environ['HOME']}/search_result_httpurls.txt"
cmd = f"seppo_nisar_search --track {track} --frame {frame} --https -o {out}"
print(cmd)

In [ ]:
nisar_search._main(cmd)

In [ ]:
! cat $HOME/search_result_httpurls.txt

# 2. Data Inspection
## Let's inspect the first link and available VARS 

The `-lg|--list_grids` picks the first url if a list is provided with the `-i <URL_LIST` flag. If you want to inspect a specific url, you can provide the url directly to the `-i <URL>` flag

In [ ]:
cmd = f"seppo_nisar_gcov_convert -lg -i {out}"
print(cmd)

In [ ]:
nisar_gcov_convert._main(cmd)

# 3. Data Processing
## Lets pick a small subset to generate the time series for 


We can do this 
- with `--srcwin <XOFF> <YOFF> <XSIZE> <YSIZE>`
- with `--projwin <ULX} <ULY> <LRX> <LRY>` in the native EPSG Coordinates
- with `--projwin` in Lon/Lat coordinates using `-t_srs 4326`. Optionally also set the target resolution e.g. `-tr 0.0002 0.0002`

We are interested in scaling the output to amplitude (`-amp` flag).

We also want to output the data on our s3:// bucket for direct streaming into QGIS later

### Example lon/lat subset to local disk

In [ ]:
projwin = "-72 46 -71.8 45.8"
tr= "0.0002 0.0002"
t_srs = 4326
scaling = "-amp"
verbose = "-v"
# Local output
output=f"{os.environ["HOME"]}/openSEPPO_testoutput{scaling}"
# S3 output
output=f"s3://seppo1-data/NISAR/openSEPPO_testoutput{scaling}"
cmd = f"seppo_nisar_gcov_convert -i {out} --projwin {projwin} -t_srs {t_srs} -tr {tr} -o {output} {scaling} {verbose}"
print(cmd)

In [ ]:
nisar_gcov_convert._main(cmd)

# Display COGs in QGIS

To Display the converted data in QGIS simply choose the `Protocol AWS s3` option, fill in bucket name and one of the VRTs object path from the final output. If you are interested in a simple Timeseries interactive click/plot tool, install from zip our Timeseries SAR plugin from https://github.com/EarthBigData/openSAR/tree/master/code/QGIS/v3/plugins